**Note:** file paths in this notebook have been replaced with `/path/to/...` placeholders. Edit them to match your own system before running.

In [ ]:
# First, generate distance files

import os
from brainsmash.workbench.geo import cortex
from brainsmash.mapgen.base import Base
import nibabel as nib
import numpy as np
from brainsmash.mapgen.memmap import txt2memmap
from brainsmash.mapgen.stats import nonparp
from brainsmash.mapgen.stats import spearmanr, pairwise_r
from scipy import stats
from brainsmash.mapgen.sampled import Sampled
import pandas as pd

# Function to process the cortex if the output file does not exist
def process_if_not_exists(surface_file, output_file):
    if not os.path.exists(output_file):
        cortex(surface=surface_file, outfile=output_file, euclid=False)

# Define your surface and output files
surface_files = [
    "/path/to/desktop/inflated_brains/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii",
    "/path/to/desktop/inflated_brains/S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii"
]
output_files = [
    "/path/to/desktop/inflated_brains/LDenseGeodesicDistmat.txt",
    "/path/to/desktop/inflated_brains/RDenseGeodesicDistmat.txt"
]

# Process each surface file
for surface, output in zip(surface_files, output_files):
    process_if_not_exists(surface, output)


In [ ]:
# Generate parcellated distance matrices

from brainsmash.workbench.geo import parcellate
infile  = output_files[0]
outfile = "/path/to/desktop/inflated_brains/LDenseGeodesicDistmat_parc.txt"
dlabel  = "/path/to/desktop/Schaefer2018_400Parcels_7Networks_order_L.dlabel.nii"
parcellate(infile, dlabel, outfile)

infile  = output_files[1]
outfile = "/path/to/desktop/inflated_brains/RDenseGeodesicDistmat_parc.txt"
dlabel  = "/path/to/desktop/Schaefer2018_400Parcels_7Networks_order_R.dlabel.nii"
parcellate(infile, dlabel, outfile)

In [ ]:
# Set up distances

distance_files = [
    "/path/to/desktop/inflated_brains/LDenseGeodesicDistmat_parc.txt",
    "/path/to/desktop/inflated_brains/RDenseGeodesicDistmat_parc.txt"
]

lh_dist = distance_files[0]
rh_dist = distance_files[1]

output_files_lh = txt2memmap(lh_dist, '/path/to/data/isc_heritability/data/smash/', maskfile=None, delimiter=' ')
output_files_rh = txt2memmap(lh_dist, '/path/to/data/isc_heritability/data/smash/', maskfile=None, delimiter=' ')


In [ ]:
# Similarity of day 1/day 2 ISC heritability maps

brain_files = [
    "/path/to/data/isc_heritability/data/anatomical/outputs/gray/anatomical_isc_herit_gray_scan_2.dscalar.nii",
    "/path/to/data/isc_heritability/data/anatomical/outputs/gray/anatomical_isc_herit_gray_scan_1.dscalar.nii"
]

process_brain_files(brain_files[0],
                    brain_files[1],
                    output_files_lh,
                    output_files_rh,
                    "/path/to/data/isc_heritability/data/smash/",1000)

In [ ]:
# Similarity of day 1/day 2 response-aligned ISC heritability maps

brain_files = [
    "/path/to/data/isc_heritability/data/piecewise/outputs/gray/piecewise_herit_diff_scan_1.dscalar.nii",
    "/path/to/data/isc_heritability/data/piecewise/outputs/gray/piecewise_herit_diff_scan_2.dscalar.nii"
]

process_brain_files(brain_files[0],
                    brain_files[1],
                    output_files_lh,
                    output_files_rh,
                    "/path/to/data/isc_heritability/data/smash/",1000)

In [ ]:
# Similarity of day 1/day 2 connectivity-aligned ISC heritability maps


brain_files = [
    "/path/to/data/isc_heritability/data/connectivity/outputs/gray/connectivity_herit_diff_scan_1.dscalar.nii",
    "/path/to/data/isc_heritability/data/connectivity/outputs/gray/connectivity_herit_diff_scan_2.dscalar.nii"
]

process_brain_files(brain_files[0],
                    brain_files[1],
                    output_files_lh,
                    output_files_rh,
                    "/path/to/data/isc_heritability/data/smash/",1000)


In [ ]:
# Similarity of day 1 response- and connectivity-aligned ISC heritability maps

brain_files = [
    "/path/to/data/isc_heritability/data/connectivity/outputs/gray/connectivity_herit_diff_scan_1.dscalar.nii",
    "/path/to/data/isc_heritability/data/piecewise/outputs/gray/piecewise_herit_diff_scan_1.dscalar.nii"
]

process_brain_files(brain_files[0],
                    brain_files[1],
                    output_files_lh,
                    output_files_rh,
                    "/path/to/data/isc_heritability/data/smash/",1000)


In [ ]:
# Similarity of day 2 response- and connectivity-aligned ISC heritability maps

brain_files = [
    "/path/to/data/isc_heritability/data/connectivity/outputs/gray/connectivity_herit_diff_scan_2.dscalar.nii",
    "/path/to/data/isc_heritability/data/piecewise/outputs/gray/piecewise_herit_diff_scan_2.dscalar.nii"
]

process_brain_files(brain_files[0],
                    brain_files[1],
                    output_files_lh,
                    output_files_rh,
                    "/path/to/data/isc_heritability/data/smash/",1000)

In [ ]:
# Run brain smashing for frequency-dependent analyses

from statsmodels.stats.multitest import multipletests
import numpy as np
import os

# Initialize pval_avg array with dimensions for days (2) and smoothing values (6)
pval_avg = np.zeros((2, 6))

# Loop over days and smoothing values
for day in range(1, 3):  # days 1 to 2
    for smooth_val in range(1, 7):  # smoothing values 1 to 6
        
        # Construct the second brain file path dynamically
        second_brain_file = f"/path/to/data/isc_heritability/data/anatomical/outputs/parc/smooth_isc_{smooth_val}_day{day}.pscalar.nii"
        
        # Define the list of brain files
        brain_files = [
            "/path/to/data/isc_heritability/data/anatomical/outputs/parc/parcel_hierarchy.pscalar.nii",
            second_brain_file
        ]

        # Define the distance files
        distance_files = [
            "/path/to/desktop/inflated_brains/LDenseGeodesicDistmat_parc.txt",
            "/path/to/desktop/inflated_brains/RDenseGeodesicDistmat_parc.txt"
        ]

        # Set output directory path
        output_path = "/path/to/data/isc_heritability/data/smash/"

        # Call the function and capture the average p-value and output CSV path
        pval, output_csv = process_brain_files_parc(brain_files, distance_files, output_path, num_surrogates=1000, n_jobs=16,seed_val=1)
        
        # Store the average p-value in the array at the appropriate index
        pval_avg[day - 1, smooth_val - 1] = pval
        
        print(f"Output CSV file saved at: {output_csv}")

# Optionally, save the pval_avg array to a file if needed
np.save(os.path.join(output_path, "pval_avg.npy"), pval_avg)

# FDR correction and significance threshold
fdr_threshold = 0.05
pval_avg1 = pval_avg[:, 0:5]

# Initialize a boolean array to store significance for each day
significant_pvals = np.zeros(pval_avg1.shape, dtype=bool)
corrected_pvals = np.zeros(pval_avg1.shape, dtype=float)
a = np.zeros(pval_avg1.shape, dtype=float)
b = np.zeros(pval_avg1.shape, dtype=float)
c = np.zeros(pval_avg1.shape, dtype=float)

# Perform FDR correction for each day independently
for day in range(2):
    # Get the p-values for the current day
    pvals_day = pval_avg[day, 0:5]
    
    # Apply FDR correction
    a[day,:], corrected_pvals[day,:], b[day,:], c[day,:] = multipletests(pvals_day, alpha=fdr_threshold, method='fdr_bh')
    
    # Store whether each smoothing value is significant (True if FDR-corrected p < 0.05)
    significant_pvals[day, :] = corrected_pvals[day, :] < fdr_threshold

# Find smoothing values that are significant on both days
significant_both_days = np.all(significant_pvals, axis=0)

# Report the smoothing values that are significant on both days
significant_smoothing_values = np.where(significant_both_days)[0] + 1  # Convert to 1-based indexing

print("Smoothing values significant on both days at FDR-corrected p < 0.05:", significant_smoothing_values)


In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from brainsmash.mapgen.base import Base

def get_surrogates(brain_files, distance_files, output_path, num_surrogates=1000, n_jobs=16, seed_val=1):
    """
    Processes brain files to generate surrogate brain maps,
    concatenates left and right hemisphere surrogates for each permutation,
    and saves the result as a CSV file where each column is a permutation
    and each row is a concatenation of the left and right hemisphere surrogates.
    
    Parameters:
        brain_files (list of str): Paths to the two pscalar brain files to process.
                                   [brain_file1, brain_file2]
        distance_files (list of str): Paths to the corresponding left and right hemisphere distance matrices.
                                      [distance_file_lh, distance_file_rh]
        output_path (str): Directory path where the output CSV file will be saved.
        num_surrogates (int): Number of surrogates to generate. Default is 1000.
        n_jobs (int): Number of jobs to run in parallel for Base class. Default is 16.
        seed_val (int): Seed value for random number generator.
        
    Returns:
        str: Path to the saved CSV file containing the concatenated surrogates.
    """
    
    # Load brain files
    cifti_file1 = nib.load(brain_files[0])  # Primary data for surrogate generation
    data1 = cifti_file1.get_fdata()
    
    # Flatten the data array to 1D
    data1 = data1.flatten()
    
    # Determine the number of parcels (vertices)
    num_parcels = data1.shape[0]
    
    # Split data into left and right hemisphere data
    # Assuming left and right hemispheres are equally divided
    lh_size = num_parcels // 2
    rh_size = num_parcels - lh_size
    
    lh_data1 = data1[:lh_size]
    rh_data1 = data1[lh_size:]
    
    # Initialize Base objects for left and right hemispheres
    base_lh = Base(x=lh_data1, D=distance_files[0], n_jobs=n_jobs, seed=seed_val,resample=False)
    base_rh = Base(x=rh_data1, D=distance_files[1], n_jobs=n_jobs, seed=seed_val,resample=False)
    
    # Generate surrogates
    surrogates_lh = base_lh(n=num_surrogates)  # Shape: (lh_size, num_surrogates)
    surrogates_rh = base_rh(n=num_surrogates)  # Shape: (rh_size, num_surrogates)
    
    # Concatenate left and right hemisphere surrogates for each permutation
    # Resulting shape: (num_parcels, num_surrogates)
    surrogates_concat = np.hstack((surrogates_lh, surrogates_rh))
    
    # Convert the array into a pandas DataFrame
    df_surrogates = pd.DataFrame(surrogates_concat)
    
    # Prepare CSV filename and path
    csv_filename = os.path.basename(brain_files[0]).replace('.pscalar.nii', '_surrogates.csv')
    output_csv_path = os.path.join(output_path, csv_filename)
    
    # Save the DataFrame to CSV without the index and header
    df_surrogates.to_csv(output_csv_path, index=False, header=False)
    
    print(f"Surrogate data saved to {output_csv_path}")
    return output_csv_path


In [ ]:
# Define function for parcel-level brain smashing

import os
import numpy as np
import nibabel as nib
import pandas as pd
from scipy import stats
from brainsmash.mapgen.stats import spearmanr, pairwise_r

def process_brain_files_parc(brain_files, distance_files, output_path, num_surrogates=1000, n_jobs=16,seed_val=1):
    """
    Processes brain files to compute surrogate brain map correlations and p-values,
    and saves the result as a CSV file.
    
    Parameters:
        brain_files (list of str): Paths to the two pscalar brain files to process.
        distance_files (list of str): Paths to the corresponding left and right hemisphere distance matrices.
        output_path (str): Directory path where the output CSV file will be saved.
        num_surrogates (int): Number of surrogates to generate. Default is 1000.
        n_jobs (int): Number of jobs to run in parallel for Base class. Default is 16.
    
    Returns:
        str: Path to the saved CSV file containing the average p-value.
    """
    
    # Load brain files
    cifti_file1 = nib.load(brain_files[0])
    cifti_file2 = nib.load(brain_files[1])
    data1 = cifti_file1.get_fdata()
    data2 = cifti_file2.get_fdata()
    
    # Split data into left and right hemisphere data
    lh_data1 = np.squeeze(data1[:, :data1.shape[1] // 2])
    rh_data1 = np.squeeze(data1[:, data1.shape[1] // 2:])
    lh_data2 = np.squeeze(data2[:, :data2.shape[1] // 2])
    rh_data2 = np.squeeze(data2[:, data2.shape[1] // 2:])
    
    # Initialize Base objects for left and right hemispheres
    base_lh = Base(x=lh_data1, D=distance_files[0], n_jobs=n_jobs,seed = seed_val)
    base_rh = Base(x=rh_data1, D=distance_files[1], n_jobs=n_jobs, seed = seed_val)
    
    # Generate surrogates
    surrogates_lh = base_lh(n=num_surrogates)
    surrogates_rh = base_rh(n=num_surrogates)
    
    # Calculate surrogate brain map correlations and test statistics
    surrogate_brainmap_corrs_lh = spearmanr(lh_data2, surrogates_lh).flatten()
    test_stat_lh = stats.spearmanr(lh_data1, lh_data2)[0]
    pval_lh = nonparp(test_stat_lh, surrogate_brainmap_corrs_lh)
    
    surrogate_brainmap_corrs_rh = spearmanr(rh_data2, surrogates_rh).flatten()
    test_stat_rh = stats.spearmanr(rh_data1, rh_data2)[0]
    pval_rh = nonparp(test_stat_rh, surrogate_brainmap_corrs_rh)
    
    # Calculate the average p-value
    pval_avg = (pval_lh + pval_rh) / 2
    
    # Prepare CSV filename and path
    csv_filename = os.path.basename(brain_files[1]).replace('.pscalar.nii', '.csv')
    output_csv_path = os.path.join(output_path, csv_filename)
    
    # Save the average p-value to a CSV file
    pd.DataFrame([pval_avg], columns=['Average_P_Value']).to_csv(output_csv_path, index=False)
    
    return pval_avg,output_csv_path


In [ ]:
# Define function for vertex-level brain smashing

def process_brain_files(brain_file1_path, brain_file2_path, output_files_lh, output_files_rh, output_path,num_perm):
    import os
    from brainsmash.workbench.geo import cortex
    from brainsmash.mapgen.base import Base
    import nibabel as nib
    import numpy as np
    from brainsmash.mapgen.memmap import txt2memmap
    from brainsmash.mapgen.stats import nonparp
    from brainsmash.mapgen.stats import spearmanr, pairwise_r
    from scipy import stats
    from brainsmash.mapgen.sampled import Sampled
    import pandas as pd
    # Read the CIFTI files and extract data
    cifti_file1 = nib.load(brain_file1_path)
    data1       = cifti_file1.get_data()
    lh_data1    = np.squeeze(data1[:, 0:(data1.shape[1] // 2)])
    rh_data1    = np.squeeze(data1[:, (data1.shape[1] // 2):])

    cifti_file2 = nib.load(brain_file2_path)
    data2       = cifti_file2.get_data()
    lh_data2    = np.squeeze(data2[:, 0:(data2.shape[1] // 2)])
    rh_data2    = np.squeeze(data2[:, (data2.shape[1] // 2):])

    # Generate lh surrogates
    dist_mat_mmap = output_files_lh['distmat']
    index_mmap    = output_files_rh['index']
    sampled       = Sampled(lh_data1, dist_mat_mmap, index_mmap)
    surrogates    = sampled(n=num_perm)
    
    surrogate_brainmap_corrs = spearmanr(lh_data2, surrogates).flatten()
    
    test_stat = stats.spearmanr(lh_data1, lh_data2)[0]
    pval_lh   = nonparp(test_stat, surrogate_brainmap_corrs)

    # Generate rh surrogates
    sampled    = Sampled(rh_data1, dist_mat_mmap, index_mmap)  
    surrogates = sampled(n=num_perm)
    
    surrogate_brainmap_corrs = spearmanr(rh_data2, surrogates).flatten()  
    
    test_stat = stats.spearmanr(rh_data1, rh_data2)[0]
    pval_rh   = nonparp(test_stat, surrogate_brainmap_corrs)

    # Calculate the average of pval_lh and pval_rh
    pval_avg = (pval_lh + pval_rh) / 2

    # Determine the CSV filename from brain_file1_path
    csv_filename = os.path.basename(brain_file1_path).replace('.dscalar.nii', '.csv')

    # Save the average p-values to a CSV file
    output_csv_path = os.path.join(output_path, csv_filename)
    
    # Save the average p-values to a CSV file
    output_csv_path = os.path.join(output_path, csv_filename)
    pd.DataFrame([pval_avg], columns=['Average_P_Value']).to_csv(output_csv_path, index=False)

    print(f"Average p-values saved to {output_csv_path}")